# TD 4 : de la description à la question

Cours de M1 Économie de l'innovation, Université de Picardie Jules Verne

**La question du TD.** Ce qu'une association entre deux variables permet d'affirmer et ce qu'elle ne permet pas d'affirmer.

| | |
|---|---|
| En séance (1 h, accompagné) | parties 1 et 2, relecture croisée entre binômes |
| À distance (environ 3 h 30) | parties 3 à 6 et finalisation du projet |
| Jalon du projet | le projet final |
| Ensuite | le projet final |

### Comment travailler avec ce notebook

Chaque partie porte un niveau. Avancez à votre rythme.

- **Essentiel** : à faire par tout le monde. Ces parties suffisent pour le projet.
- **Pour aller plus loin** : pour celles et ceux qui sont à l'aise avec Python.
- **Bonus** : des outils plus avancés (régression, probit, téléchargement direct). Les utiliser dans le projet est un plus. Ne pas les utiliser n'est jamais pénalisé.

Une cellule de code s'exécute en cliquant dessus puis en appuyant sur **Maj + Entrée**. Les zones **À vous** sont à compléter : une ligne de code ou quelques phrases. Une erreur n'est pas grave : lisez la dernière ligne du message, corrigez et relancez la cellule.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

base = pd.read_csv("donnees/base-innovation-pays-2000-2023.csv")

## 1. Formuler une question testable (Essentiel)

Une bonne question part d'un mécanisme du cours et en déduit ce que l'on devrait observer (cf. Chapitre 13). Exemple :

- **mécanisme** : la R&D produit des connaissances que les entreprises protègent en partie par des brevets (cf. Chapitre 9),
- **prédiction** : les pays qui consacrent une plus grande part de leur PIB à la R&D déposent davantage de brevets par habitant.e,
- **variables** : `rd_pib` et `brevets_par_million`.

**À vous.** Écrivez sur ce modèle le mécanisme, la prédiction et les variables de votre projet.

*Votre réponse :*

## 2. Ce qu'une corrélation ne prouve pas (Essentiel)

Une corrélation entre la R&D et les brevets peut avoir trois origines autres qu'un effet de la R&D :

- **un facteur commun** : les pays riches dépensent davantage en R&D et déposent davantage de brevets pour d'autres raisons (taille du marché, institutions),
- **une causalité inverse** : un pays où l'on brevète beaucoup attire des investissements de R&D,
- **des différences de pratiques** : la propension à breveter varie selon les secteurs et les pays.

On regarde d'abord si la relation résiste au revenu.

In [ ]:
b2021 = base[base["annee"] == 2021].dropna(subset=["rd_pib", "brevets_par_million", "pib_par_habitant"])
b2021[["rd_pib", "brevets_par_million", "pib_par_habitant"]].corr().round(2)

**À vous.** Pour votre propre question, nommez un facteur commun possible et dites si vos données permettent d'en tenir compte.

*Votre réponse :*

## 3. Une régression simple (Pour aller plus loin)

Une régression trace la droite qui résume le mieux le nuage de points. Le **coefficient** indique de combien varie en moyenne la variable expliquée quand la variable explicative augmente d'une unité.

Comme les brevets par habitant.e varient de quelques dizaines à plusieurs milliers, on prend leur logarithme. Le coefficient se lit alors en pourcentage. Pour une petite valeur, la lecture est directe : 0,05 signifie qu'un point de PIB de R&D en plus va de pair avec environ 5 % de brevets en plus. Pour une valeur plus grande, on calcule exp(coefficient) - 1 : un coefficient de 0,5 correspond à environ 65 %.

In [ ]:
import statsmodels.formula.api as smf

b2021 = b2021.assign(log_brevets=np.log(b2021["brevets_par_million"]),
                     log_pib=np.log(b2021["pib_par_habitant"]))

modele1 = smf.ols("log_brevets ~ rd_pib", data=b2021).fit()
print(modele1.summary().tables[1])
print("R² :", round(modele1.rsquared, 2), "  Nombre de pays :", int(modele1.nobs))

In [ ]:
plt.scatter(b2021["rd_pib"], b2021["log_brevets"])
x = np.linspace(b2021["rd_pib"].min(), b2021["rd_pib"].max(), 50)
plt.plot(x, modele1.params["Intercept"] + modele1.params["rd_pib"] * x)
plt.title("R&D et brevets en 2021, droite de régression")
plt.xlabel("R&D (% du PIB)")
plt.ylabel("Log des brevets par million d'habitant.es")
plt.show()

**À vous.** Interprétez le coefficient de `rd_pib` en une phrase. Indiquez s'il est statistiquement différent de zéro (colonne `P>|t|` inférieure à 0,05). Rappelez combien de pays servent à l'estimer et ce que cela implique.

*Votre réponse :*

## 4. Ajouter une variable de contrôle (Pour aller plus loin)

On ajoute le revenu. Le coefficient de `rd_pib` compare alors des pays de revenu comparable.

In [ ]:
modele2 = smf.ols("log_brevets ~ rd_pib + log_pib", data=b2021).fit()
print(modele2.summary().tables[1])

**À vous.** Comparez le coefficient de `rd_pib` dans les deux modèles. Dites ce que sa variation suggère sur le rôle du revenu. Dites pourquoi ce résultat ne prouve toujours pas un effet causal.

*Votre réponse :*

## 5. Outils avancés (Bonus)

Ces outils ne sont pas exigés pour le projet. Ils montrent comment aller plus loin.

**Données de panel et effets fixes.** En utilisant toutes les années, on peut comparer chaque pays à lui-même au cours du temps. Les effets fixes pays retirent ce qui ne change pas dans un pays (institutions, spécialisation).

In [ ]:
panel = base.dropna(subset=["rd_pib", "brevets_par_million"]).copy()
panel["log_brevets"] = np.log(panel["brevets_par_million"])
panel["pays_id"] = pd.factorize(panel["code"])[0]   # un numéro par pays
modele_fe = smf.ols("log_brevets ~ rd_pib + C(pays_id) + C(annee)", data=panel).fit(
    cov_type="cluster", cov_kwds={"groups": panel["code"]})
print("Coefficient de rd_pib :", round(modele_fe.params["rd_pib"], 2),
      "  p-valeur :", round(modele_fe.pvalues["rd_pib"], 3))

**Probit et logit.** Quand la variable à expliquer vaut 0 ou 1 (une entreprise innove ou non, un pays dépasse ou non un seuil), une régression linéaire peut prédire des probabilités inférieures à 0 ou supérieures à 1. Le probit et le logit gardent la probabilité entre 0 et 1. Avec les microdonnées de l'enquête CIS, on expliquerait ainsi la probabilité qu'une entreprise innove selon sa taille et son secteur.

L'exemple ci-dessous explique la probabilité qu'un pays-année dépasse 2 % de R&D par son revenu. Il illustre l'outil sans prétendre établir un effet.

In [ ]:
panel_b = base.dropna(subset=["rd_pib", "pib_par_habitant"]).copy()
panel_b["intensite_forte"] = (panel_b["rd_pib"] >= 2).astype(int)
panel_b["log_pib"] = np.log(panel_b["pib_par_habitant"])

probit = smf.probit("intensite_forte ~ log_pib", data=panel_b).fit(disp=False)
print(probit.get_margeff().summary())

**À vous (Bonus).** L'effet marginal (`dy/dx`) indique de combien la probabilité varie quand le log du PIB par habitant augmente d'une unité. Interprétez-le. Expliquez pourquoi les observations d'un même pays ne sont pas indépendantes d'une année à l'autre.

*Votre réponse :*

## 6. Limites et finalisation (Essentiel)

Avant de déposer le projet, vérifiez chaque point :

- [ ] la question part d'un mécanisme du cours,
- [ ] la source, la période, l'unité d'observation et les valeurs manquantes sont décrites,
- [ ] chaque graphique a un titre, des unités et une source,
- [ ] chaque résultat est commenté en lien avec le cours,
- [ ] une section dit ce que les données permettent d'affirmer et ce qu'elles ne permettent pas d'affirmer,
- [ ] le notebook s'exécute de haut en bas sans erreur (menu Kernel, Restart & Run All).

Une analyse descriptive bien construite et bien interprétée suffit pour un très bon projet. Une régression mal interprétée n'apporte rien.

---
Données : Banque mondiale, World Development Indicators, téléchargées en novembre 2026. Fichiers dans le dossier `donnees` (description dans `donnees/LISEZMOI.md`).